<a href="https://sigmoidal.ai">
  <img src="https://raw.githubusercontent.com/carlosfab/blog-sigmoidal/main/_assets/logo_sigmoidal.png" alt="Sigmoidal" width="300">
</a>

# Geometria da Formacao de Imagens

**Autor:** Carlos Melo — [sigmoidal.ai](https://sigmoidal.ai)

Neste notebook, vamos implementar os conceitos de **geometria da formacao de imagens** usando Python.

Vamos criar uma classe `Camera` que permite aplicar translacoes e rotacoes em uma matriz de projecao 3D, e visualizar os sistemas de coordenadas do mundo e da camera.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.linalg import expm

## Classe Camera e Matriz de Rotacao

A classe `Camera` armazena a matriz de projecao e oferece metodos para translacao e rotacao.

A funcao `rotation_matrix` converte um vetor de rotacao em uma matriz 4x4 usando a exponencial de matriz.

In [ ]:
class Camera:
    def __init__(self, P):
        self.P = P
        self.K = None
        self.R = None
        self.t = None
        self.c = None

    def translate(self, translation_vector):
        T = np.eye(4)
        T[:3, 3] = translation_vector
        self.P = np.dot(self.P, T)

    def rotate(self, rotation_vector):
        R = rotation_matrix(rotation_vector)
        self.P = np.dot(self.P, R)

    def position(self):
        return self.P[:3, 3]


def rotation_matrix(a):
    R = np.eye(4)
    R[:3, :3] = expm(
        np.array([[0, -a[2], a[1]],
                  [a[2], 0, -a[0]],
                  [-a[1], a[0], 0]]))
    return R

## Funcoes de Visualizacao

A funcao `draw_axis` desenha os eixos X, Y, Z em um ponto do espaco 3D.

A funcao `plot_world_and_camera` plota as coordenadas do mundo e a posicao da camera lado a lado.

In [ ]:
def draw_axis(ax, position, label, color):
    x_arrow = np.array([1, 0, 0])
    y_arrow = np.array([0, 1, 0])
    z_arrow = np.array([0, 0, 1])
    ax.quiver(*position, *x_arrow, color=color, arrow_length_ratio=0.1)
    ax.quiver(*position, *y_arrow, color=color, arrow_length_ratio=0.1)
    ax.quiver(*position, *z_arrow, color=color, arrow_length_ratio=0.1)
    ax.text(*(position + x_arrow), f"{label}_x")
    ax.text(*(position + y_arrow), f"{label}_y")
    ax.text(*(position + z_arrow), f"{label}_z")


def plot_world_and_camera(world_coordinates, camera):
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')

    world_x, world_y, world_z = zip(world_coordinates)
    ax.scatter(world_x, world_y, world_z, c='b', marker='o',
               label='World Coordinates', s=54)

    camera_x, camera_y, camera_z = camera.position()
    ax.scatter(camera_x, camera_y, camera_z, c='r', marker='^',
               label='Camera Position', s=54)

    draw_axis(ax, camera.position(), "C", color="red")
    draw_axis(ax, world_origin, "W", color="blue")

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_zlim(-3, 3)
    ax.legend()
    plt.show()

## Criando a Camera e Plotando os Sistemas de Coordenadas

Vamos criar uma camera na origem e visualizar os sistemas de coordenadas do mundo e da camera.

In [ ]:
world_origin = (1, 1, 1)

# Configurar camera
P = np.hstack((np.eye(3), np.array([[0], [0], [0]])))
cam = Camera(P)

# Plotar os sistemas de coordenadas
plot_world_and_camera(world_origin, cam)

## Aplicando Translacao e Rotacao

Agora vamos mover a camera aplicando uma translacao de `[1, 1, 1]` e uma rotacao de 45 graus ao redor do eixo Z.

In [ ]:
# Mover a camera
translation_vector = np.array([1, 1, 1])
rotation_axis = np.array([0, 0, 1])
rotation_angle = np.radians(45)
rotation_vector = rotation_axis * rotation_angle

cam.translate(translation_vector)
cam.rotate(rotation_vector)

# Plotar apos mover a camera
plot_world_and_camera(world_origin, cam)